In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [ ]:
# Create a DataFrame from your provided data
data = {
    "Optimizer": ["SGD", "RMSprop", "Adamax", "Adam", "AdamW"],
    "ACC": [0.8391, 0.8665, 0.8810, 0.9016, 0.9790],
    "AUC": [0.9605, 0.9784, 0.9810, 0.9857, 0.9857],
    "PRE": [0.8338, 0.8724, 0.8756, 0.8997, 0.9066],
    "SP": [0.9462, 0.9562, 0.9604, 0.9672, 0.9029],
    "SN": [0.8269, 0.8560, 0.8719, 0.8936, 0.9040],
    "F1": [0.8289, 0.8534, 0.8728, 0.8959, 0.8795],
    "MCC": [0.7843, 0.8257, 0.8406, 0.8682, 0.9790]
}


In [ ]:
df = pd.DataFrame(data)

In [ ]:
# Reset matplotlib settings to default
plt.rcParams.update(plt.rcParamsDefault)

# Set Times New Roman font with fallback
plt.rcParams['font.family'] = ['Times New Roman', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

# Set global plot style with large font sizes
plt.rcParams['font.size'] = 32
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['axes.titlesize'] = 40
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 22
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['savefig.dpi'] = 1000
plt.rcParams['figure.facecolor'] = 'white'

In [ ]:
# Enhanced color palette for Optimizer techniques
optimizer_palette = {
    'SGD': '#9467BD',       # Deep Purple 
    'RMSprop': '#1F77B4',   # Bright Blue 
    'Adamax': '#FF7F0E',     # Rich Orange
    'Adam': '#2CA02C',      # Vivid Green
    'AdamW': '#D62728'     # Bold Red 
}


# Metric palette
metric_palettes = {
    'ACC': 'viridis',
    'AUC': 'plasma',
    'PRE': 'inferno',
    'SP': 'magma',
    'SN': 'cividis',
    'F1': 'cool',
    'MCC': 'cubehelix',
}

In [ ]:
# Function to save figures in both PNG and PDF
def save_figure(fig, filename):
    output_folder = "Experiment_Output_Figures/Optimizer"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    png_path = os.path.join(output_folder, f"{filename}.png")
    pdf_path = os.path.join(output_folder, f"{filename}.pdf")
    
    fig.savefig(png_path, dpi=1000, bbox_inches='tight', facecolor='white', format='png')
    fig.savefig(pdf_path, dpi=1000, bbox_inches='tight', facecolor='white', format='pdf')
    
    print(f"Saved: {png_path} and {pdf_path}")
    plt.close(fig)

In [ ]:
# Function to generate the violin plot visualization
def generate_violin_plot():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    melted_df = pd.melt(df, id_vars=['Optimizer'],
                       value_vars=metrics,
                       var_name='Metric',
                       value_name='Score')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    sns.violinplot(x='Optimizer', 
                   y='Score', 
                   hue='Optimizer',  # Added hue parameter
                   data=melted_df,
                   palette=optimizer_palette, 
                   inner='box',
                   linewidth=2, 
                   ax=ax,
                   legend=False)     # Disable legend since hue matches x
    
    ax.set_title('Distribution of Performance Scores by Optimizer',
                fontsize=44, pad=30)
    ax.set_xlabel('Optimizer', fontsize=40, labelpad=25)
    ax.set_ylabel('Score Distribution Across Metrics', fontsize=40, labelpad=25)
    ax.set_ylim(0.6, 1.1)  # Adjusted for your data range
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "optimizer_violin_plot")

In [ ]:
# Function to generate the timing comparison plot
def generate_timing_comparison():
    fig, ax = plt.subplots(figsize=(18, 10))
    sorted_df = df.sort_values(by='Training_Time', ascending=True)
    
    bars = ax.barh(sorted_df['Optimizer'], sorted_df['Training_Time'],
                  color=[optimizer_palette[opt] for opt in sorted_df['Optimizer']],
                  height=0.6)
    
    for i, optimizer in enumerate(sorted_df['Optimizer']):
        test_time = sorted_df[sorted_df['Optimizer'] == optimizer]['Testing_Time'].values[0]
        ax.text(3, i, f'Test: {test_time:.4f}s', ha='left', va='center',
                fontsize=22, color='black')
    
    ax.set_title('Training and Testing Time by Optimizer', fontsize=40, pad=30)
    ax.set_xlabel('Training Time (seconds)', fontsize=36, labelpad=25)
    ax.set_ylabel('Optimizer', fontsize=36, labelpad=25)
    
    max_time = sorted_df['Training_Time'].max()
    ax.set_xlim(0, max_time * 1.2)
    # ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "optimizer_timing_comparison")

In [ ]:
metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']

In [ ]:
melted_df = pd.melt(df, id_vars=['Optimizer'],
                   value_vars=metrics,
                   var_name='Metric',
                   value_name='Score')

# Grouped bar chart
fig, ax = plt.subplots(figsize=(20, 14))
sns.barplot(x='Metric', y='Score', hue='Optimizer',
            data=melted_df, palette=optimizer_palette,
            ax=ax, edgecolor='none')

ax.set_title('Comparison of Optimizers across Metrics', fontsize=44, pad=30)
ax.set_xlabel('Evaluation Metric', fontsize=40, labelpad=25)
ax.set_ylabel('Score', fontsize=40, labelpad=25)
ax.set_ylim(0.25, 1.0)  # Adjusted for your data range

ax.legend(title='Optimizer', title_fontsize=24, fontsize=22,
          bbox_to_anchor=(1.05, 1), loc='upper left')
# ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout(pad=3.0)
save_figure(fig, "optimizer_comparison_grouped_bar")

In [ ]:
# Radar Chart
categories = metrics
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))
for i, optimizer in enumerate(df['Optimizer']):
    values = df.loc[i, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=4, label=optimizer,
            color=optimizer_palette[optimizer])
    ax.fill(angles, values, alpha=0.1, color=optimizer_palette[optimizer])

ax.set_ylim(0.25, 1.0)  # Adjusted for your data range
plt.xticks(angles[:-1], categories, fontsize=36)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=22)
plt.title('Optimizer Comparison (Radar Chart)',
          fontsize=44, pad=40, y=1.08)

plt.tight_layout(pad=3.0)
save_figure(fig, "optimizer_radar_chart")

In [ ]:
# Heatmap
heatmap_df = df[['Optimizer'] + metrics].set_index('Optimizer')
fig, ax = plt.subplots(figsize=(18, 12))

heatmap = sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu",
                     linewidths=0.5, linecolor='white', annot_kws={'size': 26},
                     vmin=0.25, vmax=1.0)  # Adjusted for your data range

cbar = heatmap.collections[0].colorbar
cbar.set_label('Score', size=34)
cbar.ax.tick_params(labelsize=28)

ax.set_title('Optimizer Performance Heatmap', fontsize=40, pad=30)
ax.set_xlabel('Evaluation Metrics', fontsize=36, labelpad=25)
ax.set_ylabel('Optimizer', fontsize=36, labelpad=25)

plt.tight_layout(pad=3.0)
save_figure(fig, "optimizer_heatmap")

In [ ]:
generate_violin_plot()

In [ ]:
# generate_timing_comparison()

In [ ]:
print("All optimizer visualizations have been generated!")
print(f"Files are saved in the 'Optimizer' folder")